In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 3)
        self.fc.weight.data = torch.tensor([[1.0], [2.0], [-1.0]])
        self.fc.bias.data = torch.tensor([1.0, 1.0, 1.0])

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        print(probs)
        return probs

x = torch.tensor([[1.0]])
model = SimpleModel()
output = model(x)

tensor([[0.2595, 0.7054, 0.0351]], grad_fn=<SoftmaxBackward0>)


In [70]:
class RegModel(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 2)
        self.olayer = nn.Linear(2, 1)
        self.layers_outputs = {}

        with torch.no_grad():
            self.fc1.weight.copy_(torch.tensor([[0.5, -0.8, 0.3],
                                                [-0.2, 0.4, 0.9]]))
            self.fc1.bias.data.copy_(torch.tensor([0.0, 0.0]))

            self.fc2.weight.copy_(torch.tensor([[0.7, -0.5],
                                                [-0.6, 0.8]]))
            self.fc2.bias.data.copy_(torch.tensor([0.0, 0.0]))

            self.olayer.weight.data.copy_(torch.tensor([[0.9, -0.3]]))
            self.olayer.bias.data.copy_(torch.tensor([0.0]))

    def forward(self, x: torch.Tensor):
        z1 = self.fc1(x)
        self.layers_outputs["z1"] = z1.detach()
        a1 = nn.ReLU()(z1)
        self.layers_outputs["a1"] = a1.detach()
        z2 = self.fc2(a1)
        self.layers_outputs["z2"] = z2.detach()
        a2 = nn.ReLU()(z2)
        self.layers_outputs["a2"] = a2.detach()
        outputs = self.olayer(a2)
        return outputs.squeeze()

model1 = RegModel()
for param in model1.parameters():
    print(f"\n{param}")


Parameter containing:
tensor([[ 0.5000, -0.8000,  0.3000],
        [-0.2000,  0.4000,  0.9000]], requires_grad=True)

Parameter containing:
tensor([0., 0.], requires_grad=True)

Parameter containing:
tensor([[ 0.7000, -0.5000],
        [-0.6000,  0.8000]], requires_grad=True)

Parameter containing:
tensor([0., 0.], requires_grad=True)

Parameter containing:
tensor([[ 0.9000, -0.3000]], requires_grad=True)

Parameter containing:
tensor([0.], requires_grad=True)


In [71]:
x = torch.tensor([3.00, -2.00, 5.00])
y = torch.tensor([-1.00])

In [72]:
yhat = model1(x)

In [73]:
model1.layers_outputs["a1"]

tensor([4.6000, 3.1000])

In [74]:
yhat

tensor(1.5030, grad_fn=<SqueezeBackward0>)

In [75]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model1.parameters(), lr=0.1)

In [76]:
loss = criterion(yhat, y.squeeze())

In [77]:
loss

tensor(6.2650, grad_fn=<MseLossBackward0>)

In [78]:
optimizer.zero_grad()
loss.backward()

In [79]:
optimizer.step()

In [80]:
for param in model1.parameters():
    print(f"\n{param}")


Parameter containing:
tensor([[-0.4461, -0.1692, -1.2769],
        [ 0.4758, -0.0505,  2.0264]], requires_grad=True)

Parameter containing:
tensor([-0.3154,  0.2253], requires_grad=True)

Parameter containing:
tensor([[-1.3725, -1.8967],
        [-0.6000,  0.8000]], requires_grad=True)

Parameter containing:
tensor([-0.4505,  0.0000], requires_grad=True)

Parameter containing:
tensor([[ 0.0640, -0.3000]], requires_grad=True)

Parameter containing:
tensor([-0.5006], requires_grad=True)


In [81]:
yhat_new = model1(x)

In [82]:
yhat_new

tensor(-3.3531, grad_fn=<SqueezeBackward0>)

In [83]:
yhat_new - yhat

tensor(-4.8561, grad_fn=<SubBackward0>)

In [84]:
model1.layers_outputs

{'z1': tensor([-7.6997, 11.8855]),
 'a1': tensor([ 0.0000, 11.8855]),
 'z2': tensor([-22.9935,   9.5084]),
 'a2': tensor([0.0000, 9.5084])}

## II.1

### (Q22) MLP-14

In [87]:
a = torch.tensor([0, 1])
b = torch.tensor([[1, 2, 3], [4, 5, 6]])
h = torch.matmul(a, b)
h

tensor([4, 5, 6])

In [90]:
wh = torch.tensor([[0.8, -1.0],
                   [0.4, -0.6]])
bh = torch.tensor([0.0, 0.0])

wz = torch.tensor([[0.3, 0.2, 0.1],
                   [-0.4, -1.0, 0.2]])
bz = torch.tensor([0.0, 0.0, 0.0])

x = torch.tensor([3.0 , 2.0], dtype=torch.float32)
y = torch.tensor([0], dtype=torch.long)

loss_fn = nn.CrossEntropyLoss()

In [92]:
# Layer 1 (h)
h = x @ wh + bh
h = F.relu(h)

# Layer 2 (z)
z = h @ wz + bz

# Loss
loss_fn(z, y)

tensor(0.8125)

### (Q23) MLP-15

In [93]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.output = nn.Linear(2, 3)
        with torch.no_grad():
            self.output.weight.copy_(torch.tensor([[0.3, -0.4],
                                                   [0.2, -1.0],
                                                   [0.1, 0.2]], dtype=torch.float32))
            self.output.bias.zero_()

    def forward(self, x):
        h = self.hidden(x)
        h = F.relu(h)
        z = self.output(h)
        return z

torch.manual_seed(0)
model = SimpleModel()
X = torch.tensor([[3.0, 2.0]], dtype=torch.float32)
y = torch.tensor([0], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
loss_fn(model(X), y)            

tensor(1.0523, grad_fn=<NllLossBackward0>)

## II.2

### (Q24-30) MLP-16-22

In [61]:
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.datasets import FashionMNIST
import torchvision.transforms as transforms
import numpy as np
import random

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [62]:
batch_size = 512
num_epochs = 15

train_dataset = FashionMNIST('../data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)

In [63]:
class GLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor):
        a, b = x.chunk(chunks=2, dim=1)
        return a * self.sigmoid(b)

class MLP(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_dims, hidden_dims*2)
        self.bn1 = nn.BatchNorm1d(hidden_dims*2)
        self.layer2 = nn.Linear(hidden_dims, hidden_dims*2)
        self.bn2 = nn.BatchNorm1d(hidden_dims*2)
        self.layer3 = nn.Linear(hidden_dims, hidden_dims*2)
        self.bn3 = nn.BatchNorm1d(hidden_dims*2)
        self.output = nn.Linear(hidden_dims, output_dims)
        self.act_fn = GLU()

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, a=0.0, mode="fan_in",
                                        nonlinearity="relu")
                nn.init.constant_(m.bias, 0.0)


    def forward(self, x):
        x = nn.Flatten()(x)
        
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.act_fn(x)
        identity = x
        
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.act_fn(x)
        
        x = self.layer3(x)
        x = self.bn3(x)
        x = self.act_fn(x)

        x = x + identity
        out = self.output(x)

        return out

In [64]:
model = MLP(input_dims=784, hidden_dims=128, output_dims=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [65]:
for epoch in range(num_epochs):
    t_loss = 0
    t_acc = 0
    cnt = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        t_loss += loss.item()
        t_acc += (torch.argmax(outputs, 1) == y).sum().item()
        cnt += len(y)

    t_loss /= len(train_loader)
    t_acc /= cnt
    print(f"Epoch {epoch+1}/{num_epochs}, Train_Loss: {t_loss:.4f}, Train_Acc: {t_acc:.4f}")

Epoch 1/15, Train_Loss: 0.5133, Train_Acc: 0.8198
Epoch 2/15, Train_Loss: 0.3406, Train_Acc: 0.8765
Epoch 3/15, Train_Loss: 0.2916, Train_Acc: 0.8944
Epoch 4/15, Train_Loss: 0.2636, Train_Acc: 0.9035
Epoch 5/15, Train_Loss: 0.2434, Train_Acc: 0.9104
Epoch 6/15, Train_Loss: 0.2177, Train_Acc: 0.9203
Epoch 7/15, Train_Loss: 0.1998, Train_Acc: 0.9268
Epoch 8/15, Train_Loss: 0.1864, Train_Acc: 0.9327
Epoch 9/15, Train_Loss: 0.1760, Train_Acc: 0.9357
Epoch 10/15, Train_Loss: 0.1550, Train_Acc: 0.9436
Epoch 11/15, Train_Loss: 0.1455, Train_Acc: 0.9469
Epoch 12/15, Train_Loss: 0.1335, Train_Acc: 0.9516
Epoch 13/15, Train_Loss: 0.1211, Train_Acc: 0.9564
Epoch 14/15, Train_Loss: 0.1111, Train_Acc: 0.9594
Epoch 15/15, Train_Loss: 0.1053, Train_Acc: 0.9621
